# APIM ❤️ Microsoft Foundry

## AI Gateway Tokenomics lab
![flow](../../images/finops-framework.gif)

Customer-ready demo of how Azure API Management, acting as an **AI Gateway**, governs the **cost and token consumption** of many agents that share a catalog of Microsoft Foundry models.

Every **agent** gets its own APIM subscription (its identity and key). Each agent belongs to a **tier** (APIM product), and the tier policy enforces:

| Control | Policy | Result when exceeded |
|---|---|---|
| 🧭 **Model governance**: which models a tier can use | `choose` + `set-body` | premium models are **downgraded** to a cheaper model, or **denied** (403) |
| ✂️ **Output cap**: maximum `max_tokens` per call | `set-body` | the request is capped transparently |
| ⏱️ **Tokens per minute** | [`llm-token-limit`](https://learn.microsoft.com/azure/api-management/llm-token-limit-policy) | 429 + `Retry-After` |
| 📦 **Token quota** per day/month | [`llm-token-limit`](https://learn.microsoft.com/azure/api-management/llm-token-limit-policy) | 403 |
| 💵 **$ budget**: every call is priced by the gateway | [`quota-by-key`](https://learn.microsoft.com/azure/api-management/quota-by-key-policy) incremented by the call's cost | 403 |

Every call is **priced at the gateway** with a price list (named value `model-pricing`). The cost and the tokens are emitted to **Application Insights** as custom metrics with the `Agent`, `Tier` and `Model` dimensions ([`llm-emit-token-metric`](https://learn.microsoft.com/azure/api-management/llm-emit-token-metric-policy) and [`emit-metric`](https://learn.microsoft.com/azure/api-management/emit-metric-policy)). The lab deploys an **Azure Monitor workbook** for the breakdown and a lightweight **web UI** to drive the demo live.

| Agent | Tier | Models | Disallowed models | TPM | Token quota | Max output | $ budget |
|---|---|---|---|---|---|---|---|
| Customer Support Agent (`support-agent`) | 🥇 Gold | gpt-4.1, gpt-4.1-mini, gpt-4.1-nano, DeepSeek-V3.2 | n/a | 20,000 | 1M / month | 2,000 | $5.00 |
| Research Agent (`research-agent`) | 🥈 Silver | gpt-4.1-mini, gpt-4.1-nano, DeepSeek-V3.2 | downgraded to gpt-4.1-mini | 12,000 | 200K / month | 800 | $0.01 |
| Marketing Copilot (`marketing-agent`) | 🥉 Bronze | gpt-4.1-nano | denied (403) | 1,500 | 20K / day | 300 | $0.02 |

> The Silver and Bronze budgets are deliberately tiny so you can exhaust them live during a demo. Use **Reset budgets** (step 🔟 or the UI) to start over.

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/): run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Adjust the location, models, tiers and agents to your preferences. Check the [model availability by region](https://learn.microsoft.com/azure/ai-foundry/openai/concepts/models).
- ⚠️ The prices are **demo prices** in USD per 1M tokens. Check the [Azure OpenAI pricing](https://azure.microsoft.com/pricing/details/cognitive-services/openai-service/) and the Foundry Models pricing for the current prices in your region.

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "swedencentral"

aiservices_config = [{"name": "foundry1", "location": "swedencentral"}]

# Model catalog with demo prices (USD per 1M tokens) used by the gateway to price every call
models_config = [
    {"name": "gpt-4.1",       "publisher": "OpenAI",   "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50, "inputPrice": 2.00, "outputPrice": 8.00},
    {"name": "gpt-4.1-mini",  "publisher": "OpenAI",   "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50, "inputPrice": 0.40, "outputPrice": 1.60},
    {"name": "gpt-4.1-nano",  "publisher": "OpenAI",   "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50, "inputPrice": 0.10, "outputPrice": 0.40},
    {"name": "DeepSeek-V3.2", "publisher": "DeepSeek", "version": "1",          "sku": "GlobalStandard", "capacity": 50, "inputPrice": 0.58, "outputPrice": 1.68},
]

# Tiers (APIM products) and their tokenomics controls
tiers_config = [
    {"name": "gold", "displayName": "Gold", "description": "Premium agents: every model, generous limits",
     "allowedModels": ["gpt-4.1", "gpt-4.1-mini", "gpt-4.1-nano", "DeepSeek-V3.2"], "fallbackModel": "gpt-4.1-mini", "onDisallowedModel": "downgrade",
     "maxOutputTokens": 2000, "tpm": 20000, "tokenQuota": 1000000, "tokenQuotaPeriod": "Monthly", "budgetUsd": 5.00},
    {"name": "silver", "displayName": "Silver", "description": "Standard agents: cost-efficient models, premium requests are downgraded",
     "allowedModels": ["gpt-4.1-mini", "gpt-4.1-nano", "DeepSeek-V3.2"], "fallbackModel": "gpt-4.1-mini", "onDisallowedModel": "downgrade",
     "maxOutputTokens": 800, "tpm": 12000, "tokenQuota": 200000, "tokenQuotaPeriod": "Monthly", "budgetUsd": 0.01},
    {"name": "bronze", "displayName": "Bronze", "description": "Low-cost agents: nano model only, tight token limits",
     "allowedModels": ["gpt-4.1-nano"], "fallbackModel": "gpt-4.1-nano", "onDisallowedModel": "deny",
     "maxOutputTokens": 300, "tpm": 1500, "tokenQuota": 20000, "tokenQuotaPeriod": "Daily", "budgetUsd": 0.02},
]
budget_period_seconds = 30 * 24 * 3600 # the $ budgets renew every 30 days

# Agents (APIM subscriptions) - each agent belongs to one tier
agents_config = [
    {"name": "support-agent",   "displayName": "Customer Support Agent", "tier": "gold"},
    {"name": "research-agent",  "displayName": "Research Agent",         "tier": "silver"},
    {"name": "marketing-agent", "displayName": "Marketing Copilot",      "tier": "bronze"},
]

apim_sku = 'Basicv2'
inference_api_path = "inference"  # path to the inference API in the APIM service
inference_api_type = "AzureOpenAIV1"  # OpenAI v1 compatible API: /inference/openai/v1/chat/completions
foundry_project_name = deployment_name

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

Deployed resources: Log Analytics, Application Insights (custom metrics with dimensions), API Management, Microsoft Foundry with the model deployments, the inference API with the [API policy](policy.xml), one product per tier with the [tier policy](product-policy.xml), one subscription per agent and the **AI Gateway Tokenomics** workbook.

In [ ]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Price list used by the gateway. Format: <model>=<input USD per 1M>/<output USD per 1M>;...
model_pricing = ";".join(f"{m['name']}={m['inputPrice']:.2f}/{m['outputPrice']:.2f}" for m in models_config)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": [{k: v for k, v in m.items() if k not in ("inputPrice", "outputPrice")} for m in models_config] },
        "modelPricing": { "value": model_pricing },
        "tiersConfig": { "value": [{**{k: v for k, v in t.items() if k != "budgetUsd"},
                                    "budgetMicroUsd": int(round(t["budgetUsd"] * 1_000_000)),  # budgets are tracked in micro-USD
                                    "budgetPeriodSeconds": budget_period_seconds} for t in tiers_config] },
        "agentsConfig": { "value": agents_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the outputs from the Bicep deployment and write the configuration of the demo UI (`src/demo-config.private.config`, which is git-ignored because it contains the agent keys).

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    outputs = output.json_data['properties']['outputs']
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    inference_base_url = utils.get_deployment_output(output, 'inferenceBaseUrl', 'Inference base URL')
    app_insights_id = utils.get_deployment_output(output, 'appInsightsId')
    app_insights_app_id = utils.get_deployment_output(output, 'appInsightsAppId', 'Application Insights App Id')
    workbook_id = utils.get_deployment_output(output, 'workbookId')
    agent_keys = outputs['agentKeys']['value']
    for agent in agent_keys:
        utils.print_info(f"Agent: {agent['displayName']} ({agent['name']}) | tier: {agent['tier']} | key: ****{agent['key'][-4:]}")

    demo_config = {
        "inferenceBaseUrl": inference_base_url, "apimServiceId": apim_service_id, "appInsightsId": app_insights_id,
        "appInsightsAppId": app_insights_app_id, "workbookId": workbook_id, "agents": agent_keys,
        "tiers": tiers_config, "models": [{k: m[k] for k in ("name", "inputPrice", "outputPrice")} for m in models_config],
    }
    with open(os.path.join("src", "demo-config.private.config"), "w") as f:
        json.dump(demo_config, f, indent=2)
    utils.print_ok("Demo UI configuration written to src/demo-config.private.config")

<a id='4'></a>
### 4️⃣ Smoke test: every agent calls an allowed model

`call_gateway` sends an OpenAI-compatible chat completion through APIM with the agent's key and prints what the gateway did, using the `x-gw-*` response headers set by the policies (served model, tokens, cost and remaining tokens per minute).

In [ ]:
import requests, time

agents = {agent['name']: agent for agent in agent_keys}
session_log = []

def call_gateway(agent, model, prompt, max_tokens=200, verbose=True):
    response = requests.post(f"{inference_base_url}/chat/completions", headers={"api-key": agents[agent]['key']}, timeout=120,
                             json={"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]})
    headers = response.headers
    result = {"agent": agent, "tier": agents[agent]['tier'], "status": response.status_code, "requested": model,
              "served": headers.get("x-gw-model-served", ""), "governance": headers.get("x-gw-governance", ""),
              "promptTokens": int(headers.get("x-gw-prompt-tokens", 0)), "completionTokens": int(headers.get("x-gw-completion-tokens", 0)),
              "costUsd": float(headers.get("x-gw-cost-usd", 0)), "remainingTpm": headers.get("x-gw-remaining-tpm", ""),
              "retryAfter": headers.get("Retry-After", "")}
    if response.status_code != 200:
        try:
            result["error"] = response.json().get("error", {}).get("message") or response.json().get("message") or response.text
        except ValueError:
            result["error"] = response.text
    session_log.append(result)
    if verbose:
        summary = (f"{agent} [{result['tier']}] {model} -> {result['served'] or '-'} | status {response.status_code} | governance: {result['governance'] or '-'} | "
                   f"tokens {result['promptTokens']}+{result['completionTokens']} | cost ${result['costUsd']:.6f} | remaining TPM {result['remainingTpm'] or '-'}")
        if response.status_code == 200:
            utils.print_ok(summary)
        else:
            utils.print_error(f"{summary} | {result.get('error', '')[:200]}")
    return result

call_gateway("support-agent", "gpt-4.1", "In one sentence, what is an AI gateway?")
call_gateway("support-agent", "DeepSeek-V3.2", "In one sentence, what is an AI gateway?")
call_gateway("research-agent", "gpt-4.1-mini", "In one sentence, what is an AI gateway?")
call_gateway("marketing-agent", "gpt-4.1-nano", "Give me a 5 word tagline for an AI gateway.")

<a id='5'></a>
### 5️⃣ Model governance: downgrade and deny

- The **Research Agent** (Silver) asks for the premium `gpt-4.1`. The gateway transparently **downgrades** the request to `gpt-4.1-mini` (`x-gw-governance: model-downgraded`).
- The **Marketing Copilot** (Bronze) asks for `gpt-4.1`. The gateway **denies** the request with a 403 before it reaches Foundry.

In [ ]:
prompt = "Explain in two sentences why model routing reduces AI costs."
downgraded = call_gateway("research-agent", "gpt-4.1", prompt)
denied = call_gateway("marketing-agent", "gpt-4.1", prompt)

assert downgraded["status"] == 200 and downgraded["served"] == "gpt-4.1-mini", "Expected the Silver request to be downgraded"
assert denied["status"] == 403, "Expected the Bronze request to be denied"
utils.print_ok("Model governance enforced: Silver downgraded to gpt-4.1-mini, Bronze denied")

<a id='6'></a>
### 6️⃣ Output token cap

The **Marketing Copilot** asks for 4,000 output tokens. The Bronze tier caps `max_tokens` at 300 (`x-gw-governance: output-capped`), which bounds the worst-case cost of every call.

In [ ]:
capped = call_gateway("marketing-agent", "gpt-4.1-nano", "Write a 1500-word essay about the history of marketing.", max_tokens=4000)
assert capped["status"] == 200 and capped["completionTokens"] <= 300 and "output-capped" in capped["governance"]
utils.print_ok(f"Output capped: {capped['completionTokens']} completion tokens (requested 4000, Bronze cap 300)")

<a id='7'></a>
### 7️⃣ Tokens per minute: burst the Bronze agent

The Bronze tier allows 1,500 tokens per minute. A burst of requests is throttled with **429** and a `Retry-After` header.

In [ ]:
burst = [call_gateway("marketing-agent", "gpt-4.1-nano", "Write a 500-word product description for a smart water bottle.", max_tokens=300) for _ in range(10)]
throttled = [r for r in burst if r["status"] == 429]
assert throttled, "Expected at least one 429"
utils.print_ok(f"{len(throttled)} of {len(burst)} requests throttled with 429 (Retry-After: {throttled[0]['retryAfter']} seconds)")

<a id='8'></a>
### 8️⃣ $ budget: exhaust the Research Agent budget

The Silver tier has a **$0.01** budget. Every successful call increments the agent's `quota-by-key` counter by the call's cost in micro-USD, as computed by the gateway. When the budget is spent, the gateway answers **403** and protects you from runaway spend.

In [ ]:
spent = sum(r["costUsd"] for r in session_log if r["agent"] == "research-agent")
for i in range(30):
    result = call_gateway("research-agent", "gpt-4.1-mini", "Write a detailed report about the evolution of large language models.", max_tokens=800)
    spent += result["costUsd"]
    if result["status"] == 429:
        time.sleep(int(result["retryAfter"] or 10))
    if result["status"] == 403:
        break

assert result["status"] == 403, "Expected the budget to be exhausted"
utils.print_ok(f"Budget exhausted after spending ${spent:.6f} of the $0.01 Silver budget")

<a id='9'></a>
### 9️⃣ Tokenomics in Application Insights

The gateway emits the `Prompt Tokens`, `Completion Tokens`, `Total Tokens`, `CostMicroUSD` and `GovernanceEvents` custom metrics with the `Agent`, `Tier` and `Model` dimensions. The APIM request logs, with the subscription (agent) and product (tier), show the 403 and 429 outcomes.

> ⏳ Telemetry takes 1-3 minutes to arrive in Application Insights.

In [ ]:
import pandas as pd

def query_app_insights(query, timespan="PT4H"):
    token = utils.run("az account get-access-token --resource https://api.applicationinsights.io", print_command_to_run=False).json_data["accessToken"]
    response = requests.post(f"https://api.applicationinsights.io/v1/apps/{app_insights_app_id}/query", headers={"Authorization": f"Bearer {token}"},
                             json={"query": query, "timespan": timespan}, timeout=60)
    response.raise_for_status()
    table = response.json()["tables"][0]
    return pd.DataFrame(table["rows"], columns=[c["name"] for c in table["columns"]])

dims = '| extend Agent = tostring(customDimensions["Agent"]), Tier = tostring(customDimensions["Tier"]), Model = tostring(customDimensions["Model"])'
cost_query = f'customMetrics | where name == "CostMicroUSD" {dims} | summarize CostUSD = round(sum(valueSum) / 1e6, 6), Calls = sum(valueCount) by Agent, Tier, Model | order by CostUSD desc'
tokens_query = (f'customMetrics | where name in ("Prompt Tokens", "Completion Tokens") {dims} '
                '| summarize PromptTokens = sumif(valueSum, name == "Prompt Tokens"), CompletionTokens = sumif(valueSum, name == "Completion Tokens") by Agent, Model')
outcomes_query = ('requests | where url has "/openai/" | extend Agent = tostring(customDimensions["Subscription Name"]), Tier = tostring(customDimensions["Product Name"]) '
                  '| summarize Calls = count() by Agent, Tier, resultCode | order by Agent asc, resultCode asc')

for attempt in range(12):
    cost_df = query_app_insights(cost_query)
    if not cost_df.empty:
        break
    utils.print_info("Waiting for telemetry to arrive in Application Insights...")
    time.sleep(20)

utils.print_ok("Cost by agent and model (from Application Insights)")
display(cost_df)
utils.print_ok("Tokens by agent and model")
display(query_app_insights(tokens_query))
utils.print_ok("Gateway outcomes by agent")
display(query_app_insights(outcomes_query))

#### 📊 Visualize the cost breakdown

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cost_df.groupby("Agent")["CostUSD"].sum().sort_values().plot(kind="barh", ax=axes[0], color="#4f8cff", title="Cost by agent (USD)")
cost_df.groupby("Model")["CostUSD"].sum().plot(kind="pie", ax=axes[1], autopct="%1.1f%%", title="Cost by model")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

workbook_url = f"https://portal.azure.com/#@/resource{workbook_id}/workbook"
utils.print_info(f"Open the AI Gateway Tokenomics workbook for the full breakdown: {workbook_url}")

<a id='10'></a>
### 🔟 Launch the demo UI

The lightweight web UI ([src/app.py](src/app.py), Python standard library only) lets you pick an agent and a model, send single requests, bursts or mixed traffic, see what the gateway did for every call, track each agent's $ budget, and query the Application Insights breakdown live.

Run it from a terminal in this folder to keep it running independently of the notebook:

```bash
python src/app.py --port 8080
```

Or run the next cell to start it in the background, then open http://localhost:8080.

In [ ]:
import subprocess
ui_process = subprocess.Popen([sys.executable, os.path.join("src", "app.py"), "--port", "8080"])
time.sleep(2)
utils.print_ok("Demo UI running on http://localhost:8080 (run ui_process.terminate() to stop it)")

<a id='11'></a>
### ↺ Reset the budgets and token quotas

Every counter key includes the `budget-epoch` named value, so updating it starts fresh budgets and token quotas for every agent. Run this before each demo. The UI **Reset budgets** button does the same.

In [ ]:
epoch = str(int(time.time()))
output = utils.run(f"az apim nv update --service-name {apim_service_id.split('/')[-1]} -g {resource_group_name} --named-value-id budget-epoch --value {epoch}",
                   "Budgets and token quotas reset", "Failed to reset the budgets")

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.